# GrandStay Hospitality Group (GSH)
## Hospitality Operations Intelligence Initiative
### Data Analysis and Business Case for AI Travel Concierge

---

**Prepared by:** Oba Adelore  
**Role:** Assistant Group Leader  
**Programme:** Amdari Analytics Programme  
**Date:** May 2026  
**Tools:** Python, Pandas, Matplotlib, Seaborn, Scikit-learn

---

### Project Overview

GrandStay Hospitality Group (GSH) is a globally recognised hospitality enterprise operating 8,000+ properties across 130 countries. This notebook delivers a complete data-driven analysis of GSH's guest support operations, building a defensible business case for the implementation of an Intelligent Travel Concierge.

### Objectives

1. **Quantify Operational Inefficiencies** across Time, Scale and Cost dimensions  
2. **Identify Automation Opportunities** from repetitive inquiry patterns  
3. **Evaluate Service Reliability** — SLA adherence, response consistency across regions and channels  
4. **Model Financial Impact** — cost savings, revenue recovery and ROI  
5. **Establish Performance Benchmarks** — baseline KPIs for pre and post-automation evaluation  
6. **Design Measurement Framework** — structured post-automation monitoring approach

### Datasets

| Dataset | Records | Description |
|---------|---------|-------------|
| Agent_Performance_Logs | 120,000 | Ticket-level service metrics |
| Cost_Efficiency_Metrics | 12 | Monthly aggregated financial data |
| Inquiry_Volume_Patterns | 43,125 | Hourly inquiry trends by region and channel |
| Knowledge_Escalation_Logs | 5,000 | Knowledge base usage and escalation behaviour |


---
## Section 1: Environment Setup and Data Loading


In [1]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set consistent visual style
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Colour palette aligned with GSH dashboard theme
NAVY = '#1F4E79'
GOLD = '#C9A84C'
LIGHT_BLUE = '#BDD7EE'
MID_BLUE = '#2E75B6'
RED = '#C00000'
GREEN = '#70AD47'
ORANGE = '#E36C09'
YELLOW = '#F0C000'
GREY = '#BFBFBF'

GSH_PALETTE = [NAVY, MID_BLUE, LIGHT_BLUE, GOLD, GREEN, ORANGE, RED]

print("Libraries loaded successfully.")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")


Libraries loaded successfully.
Pandas version: 2.2.3
NumPy version: 2.1.3


In [2]:
# Load all four datasets
agent = pd.read_excel('agent_performance_logs.xlsx')
cost = pd.read_excel('cost_efficiency_metrics.xlsx')
inquiry = pd.read_excel('inquiry_volume_patterns.xlsx')
knowledge = pd.read_excel('knowledge_escalation_logs.xlsx')

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
for name, df in [('Agent Performance Logs', agent), 
                  ('Cost Efficiency Metrics', cost),
                  ('Inquiry Volume Patterns', inquiry), 
                  ('Knowledge Escalation Logs', knowledge)]:
    print(f"\n{name}")
    print(f"  Rows: {len(df):,} | Columns: {len(df.columns)}")
    print(f"  Columns: {list(df.columns)}")


FileNotFoundError: [Errno 2] No such file or directory: 'agent_performance_logs.xlsx'

---
## Section 2: Data Cleaning and Preparation

This section handles all data cleaning tasks:
- Converting date serial numbers to proper datetime formats
- Computing derived metrics (Response_Time_Min, Handle_Time_Min, SLA_Breach)
- Handling missing CSAT scores
- Creating readable category labels for boolean columns


In [ ]:
# ── AGENT LOGS: Date conversion ──────────────────────────────────────────────
agent['created_time'] = pd.to_datetime(agent['created_time'])
agent['first_response_time'] = pd.to_datetime(agent['first_response_time'])
agent['resolved_time'] = pd.to_datetime(agent['resolved_time'])
agent['date'] = pd.to_datetime(agent['date'])
agent['month'] = pd.to_datetime(agent['month'])

# ── AGENT LOGS: Derived columns ───────────────────────────────────────────────
agent['Response_Time_Min'] = (
    agent['first_response_time'] - agent['created_time']
).dt.total_seconds() / 60

agent['Handle_Time_Min'] = (
    agent['resolved_time'] - agent['first_response_time']
).dt.total_seconds() / 60

agent['SLA_Breach'] = agent['Response_Time_Min'] > agent['sla_target_min']

# ── AGENT LOGS: Readable labels ───────────────────────────────────────────────
agent['Inquiry_Type'] = agent['is_repetitive'].map({True: 'Repetitive', False: 'Non Repetitive'})
agent['SLA_Status'] = agent['SLA_Breach'].map({True: 'Breached', False: 'Met'})
agent['Peak_Label'] = agent['is_peak_hour'].map({True: 'Peak Hour', False: 'Off Peak'})
agent['CSAT_Available'] = agent['csat_score'].notna().map({True: 'Yes', False: 'No'})

# ── COST METRICS: Date conversion ─────────────────────────────────────────────
cost['month'] = pd.to_datetime(cost['month'])
cost['Month_Label'] = cost['month'].dt.strftime('%b-%y')

# ── INQUIRY PATTERNS: Date conversion ────────────────────────────────────────
inquiry['date'] = pd.to_datetime(inquiry['date'])
inquiry['date_time'] = pd.to_datetime(inquiry['date_time'])

# ── KNOWLEDGE LOGS: Date conversion and labels ────────────────────────────────
knowledge['date'] = pd.to_datetime(knowledge['date'])
knowledge['Escalation_Status'] = knowledge['was_escalated'].map(
    {True: 'Escalated', False: 'Not Escalated'}
)

print("Data cleaning complete.")
print(f"\nAgent Logs: {len(agent):,} rows | New columns: Response_Time_Min, Handle_Time_Min, SLA_Breach, Inquiry_Type")
print(f"Missing CSAT scores: {agent['CSAT_Available'].eq('No').sum():,} out of {len(agent):,}")
print(f"\nResponse_Time_Min stats:")
print(agent['Response_Time_Min'].describe().round(2))


In [ ]:
# Data Quality Summary
print("=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)

datasets = {
    'Agent Logs': agent,
    'Cost Metrics': cost,
    'Inquiry Patterns': inquiry,
    'Knowledge Logs': knowledge
}

for name, df in datasets.items():
    missing = df.isnull().sum()
    missing_cols = missing[missing > 0]
    print(f"\n{name} ({len(df):,} rows):")
    if len(missing_cols) == 0:
        print("  No missing values (except CSAT which is handled)")
    else:
        for col, count in missing_cols.items():
            print(f"  {col}: {count:,} missing ({count/len(df)*100:.1f}%)")


---
## Section 3: Time Dimension Analysis — SLA Breaches and Response Times

**Objective:** Measure performance gaps across the Time dimension including response delays, SLA adherence and peak period identification.


In [ ]:
# ── SLA Breach Summary ────────────────────────────────────────────────────────
total_tickets = len(agent)
total_breached = agent['SLA_Breach'].sum()
breach_rate = total_breached / total_tickets * 100

print("=" * 60)
print("OVERALL SLA PERFORMANCE")
print("=" * 60)
print(f"Total Tickets:        {total_tickets:,}")
print(f"SLA Breached:         {total_breached:,}")
print(f"SLA Met:              {total_tickets - total_breached:,}")
print(f"Overall Breach Rate:  {breach_rate:.2f}%")
print(f"Average Response Time: {agent['Response_Time_Min'].mean():.2f} minutes")
print(f"Median Response Time:  {agent['Response_Time_Min'].median():.2f} minutes")
print(f"Average Handle Time:   {agent['Handle_Time_Min'].mean():.2f} minutes")


In [ ]:
# ── SLA Breach by Region ─────────────────────────────────────────────────────
region_breach = agent.groupby('region').agg(
    Total_Tickets=('ticket_id', 'count'),
    Breached=('SLA_Breach', 'sum')
).reset_index()
region_breach['Breach_Rate_%'] = (region_breach['Breached'] / region_breach['Total_Tickets'] * 100).round(2)
region_breach = region_breach.sort_values('Breach_Rate_%', ascending=False)

print("SLA BREACH RATE BY REGION")
print(region_breach.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(region_breach['region'], region_breach['Breach_Rate_%'],
               color=NAVY, edgecolor='white')
ax.axvline(x=breach_rate, color=RED, linestyle='--', linewidth=1.5, label=f'Overall Average ({breach_rate:.1f}%)')
for bar, val in zip(bars, region_breach['Breach_Rate_%']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.2f}%',
            va='center', fontsize=10, color=NAVY, fontweight='bold')
ax.set_xlabel('SLA Breach Rate (%)', fontsize=11)
ax.set_title('SLA Breach Rate by Region', fontsize=14, fontweight='bold', color=NAVY, pad=15)
ax.legend(fontsize=10)
ax.set_xlim(0, 100)
plt.tight_layout()
plt.savefig('sla_breach_by_region.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nKey Finding: All 7 regions breach SLA at approximately 72% — the problem is systemic not regional.")


In [ ]:
# ── SLA Breach by Channel ────────────────────────────────────────────────────
channel_breach = agent.groupby('channel').agg(
    Total_Tickets=('ticket_id', 'count'),
    Breached=('SLA_Breach', 'sum')
).reset_index()
channel_breach['Breach_Rate_%'] = (channel_breach['Breached'] / channel_breach['Total_Tickets'] * 100).round(2)
channel_breach = channel_breach.sort_values('Breach_Rate_%', ascending=False)

print("SLA BREACH RATE BY CHANNEL")
print(channel_breach.to_string(index=False))

colours = [RED if r > 75 else ORANGE if r > 60 else YELLOW if r > 45 else GREEN 
           for r in channel_breach['Breach_Rate_%']]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(channel_breach['channel'], channel_breach['Breach_Rate_%'],
               color=colours, edgecolor='white')
ax.axvline(x=breach_rate, color=NAVY, linestyle='--', linewidth=1.5, label=f'Overall Average ({breach_rate:.1f}%)')
for bar, val in zip(bars, channel_breach['Breach_Rate_%']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.2f}%',
            va='center', fontsize=11, fontweight='bold', color=NAVY)
ax.set_xlabel('SLA Breach Rate (%)', fontsize=11)
ax.set_title('SLA Breach Rate by Channel', fontsize=14, fontweight='bold', color=NAVY, pad=15)
ax.legend(fontsize=10)
ax.set_xlim(0, 110)
plt.tight_layout()
plt.savefig('sla_breach_by_channel.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nKey Finding: Phone 84.82% and Chat 77.06% are the worst performing channels and also the highest volume.")


In [ ]:
# ── SLA Breach by Hour of Day ─────────────────────────────────────────────────
hour_breach = agent.groupby('hour_of_day').agg(
    Total=('ticket_id', 'count'),
    Breached=('SLA_Breach', 'sum')
).reset_index()
hour_breach['Breach_Rate_%'] = (hour_breach['Breached'] / hour_breach['Total'] * 100).round(2)

fig, ax = plt.subplots(figsize=(14, 6))
colours_hour = [RED if (8 <= h <= 11) or (15 <= h <= 19) else MID_BLUE 
                for h in hour_breach['hour_of_day']]
ax.bar(hour_breach['hour_of_day'], hour_breach['Breach_Rate_%'], 
       color=colours_hour, edgecolor='white', width=0.8)
ax.axhline(y=breach_rate, color=GOLD, linestyle='--', linewidth=2, 
           label=f'Overall Average ({breach_rate:.1f}%)')
ax.axvspan(7.5, 11.5, alpha=0.08, color=RED, label='Morning Peak (8-11)')
ax.axvspan(14.5, 19.5, alpha=0.08, color=ORANGE, label='Afternoon Peak (15-19)')
ax.set_xlabel('Hour of Day', fontsize=11)
ax.set_ylabel('SLA Breach Rate (%)', fontsize=11)
ax.set_title('SLA Breach Rate by Hour of Day — Peak Windows Identified', 
             fontsize=14, fontweight='bold', color=NAVY, pad=15)
ax.set_xticks(range(0, 24))
ax.legend(fontsize=10)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig('sla_breach_by_hour.png', dpi=150, bbox_inches='tight')
plt.show()

peak_hours = hour_breach[hour_breach['hour_of_day'].between(8, 11) | 
                          hour_breach['hour_of_day'].between(15, 19)]['Breach_Rate_%'].mean()
offpeak_hours = hour_breach[~(hour_breach['hour_of_day'].between(8, 11) | 
                               hour_breach['hour_of_day'].between(15, 19))]['Breach_Rate_%'].mean()
print(f"Peak hour average breach rate (8-11 and 15-19): {peak_hours:.2f}%")
print(f"Off-peak average breach rate: {offpeak_hours:.2f}%")
print(f"Spike during peak windows: +{peak_hours - offpeak_hours:.1f} percentage points")


---
## Section 4: Scale Dimension Analysis — Inquiry Volume and Abandonment

**Objective:** Quantify inquiry volume growth, channel distribution and abandonment patterns.


In [ ]:
# ── Inquiry Volume by Channel ─────────────────────────────────────────────────
vol_channel = inquiry.groupby('channel')['inquiry_volume'].sum().reset_index()
vol_channel = vol_channel.sort_values('inquiry_volume', ascending=False)
vol_channel['%_of_Total'] = (vol_channel['inquiry_volume'] / vol_channel['inquiry_volume'].sum() * 100).round(2)

print("INQUIRY VOLUME BY CHANNEL")
print(vol_channel.to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart
ch_colours = [ORANGE, RED, YELLOW, GREEN]
bars = ax1.barh(vol_channel['channel'], vol_channel['inquiry_volume'],
                color=ch_colours, edgecolor='white')
for bar, val, pct in zip(bars, vol_channel['inquiry_volume'], vol_channel['%_of_Total']):
    ax1.text(val + 200, bar.get_y() + bar.get_height()/2, 
             f'{val:,} ({pct}%)', va='center', fontsize=10, color=NAVY, fontweight='bold')
ax1.set_xlabel('Total Inquiries', fontsize=11)
ax1.set_title('Inquiry Volume by Channel', fontsize=13, fontweight='bold', color=NAVY)

# Donut chart
wedges, texts, autotexts = ax2.pie(
    vol_channel['inquiry_volume'],
    labels=vol_channel['channel'],
    colors=ch_colours,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(width=0.6)
)
for autotext in autotexts:
    autotext.set_fontsize(11)
    autotext.set_fontweight('bold')
ax2.set_title('Channel Share of Total Volume', fontsize=13, fontweight='bold', color=NAVY)

plt.tight_layout()
plt.savefig('inquiry_volume_by_channel.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nChat and Phone combined: {vol_channel[vol_channel['channel'].isin(['Chat','Phone'])]['%_of_Total'].sum():.1f}% of all inquiries")


In [ ]:
# ── Inquiry Volume Growth Over Time ──────────────────────────────────────────
cost['month'] = pd.to_datetime(cost['month'])
cost_sorted = cost.sort_values('month')

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(cost_sorted['month'], cost_sorted['total_inquiries'], 
        color=MID_BLUE, linewidth=2.5, marker='o', markersize=6)
for i, row in cost_sorted.iterrows():
    ax.annotate(f"{row['total_inquiries']:,}", 
                (row['month'], row['total_inquiries']),
                textcoords="offset points", xytext=(0, 10),
                ha='center', fontsize=8.5, color=NAVY)
ax.fill_between(cost_sorted['month'], cost_sorted['total_inquiries'], 
                alpha=0.1, color=MID_BLUE)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Total Inquiries', fontsize=11)
ax.set_title('Monthly Inquiry Volume Growth — Nov 2024 to Oct 2025', 
             fontsize=14, fontweight='bold', color=NAVY, pad=15)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('inquiry_volume_growth.png', dpi=150, bbox_inches='tight')
plt.show()

growth_rate = (cost_sorted['total_inquiries'].iloc[-1] - cost_sorted['total_inquiries'].iloc[0]) / cost_sorted['total_inquiries'].iloc[0] * 100
print(f"Inquiry volume Nov 2024: {cost_sorted['total_inquiries'].iloc[0]:,}")
print(f"Inquiry volume Oct 2025: {cost_sorted['total_inquiries'].iloc[-1]:,}")
print(f"Growth rate: +{growth_rate:.1f}%")


In [ ]:
# ── Abandonment Rate by Channel and Region ────────────────────────────────────
abandon_channel = inquiry.groupby('channel')['abandonment_rate'].mean().reset_index()
abandon_channel['abandonment_rate_%'] = (abandon_channel['abandonment_rate'] * 100).round(2)
abandon_channel = abandon_channel.sort_values('abandonment_rate_%', ascending=False)

abandon_region = inquiry.groupby('region')['abandonment_rate'].mean().reset_index()
abandon_region['abandonment_rate_%'] = (abandon_region['abandonment_rate'] * 100).round(2)

print("ABANDONMENT RATE BY CHANNEL")
print(abandon_channel[['channel', 'abandonment_rate_%']].to_string(index=False))
print(f"\nGlobal Average: {inquiry['abandonment_rate'].mean()*100:.2f}%")
print(f"Total Abandoned Inquiries: {inquiry['abandoned_inquiries'].sum():,}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ch_colours = [RED if r > 12 else ORANGE if r > 10 else YELLOW if r > 8 else GREEN 
              for r in abandon_channel['abandonment_rate_%']]
bars = ax1.barh(abandon_channel['channel'], abandon_channel['abandonment_rate_%'],
                color=ch_colours, edgecolor='white')
for bar, val in zip(bars, abandon_channel['abandonment_rate_%']):
    ax1.text(val + 0.1, bar.get_y() + bar.get_height()/2, f'{val:.2f}%',
             va='center', fontsize=10, fontweight='bold', color=NAVY)
ax1.set_xlabel('Abandonment Rate (%)', fontsize=11)
ax1.set_title('Abandonment Rate by Channel', fontsize=13, fontweight='bold', color=NAVY)

bars2 = ax2.barh(abandon_region['region'], abandon_region['abandonment_rate_%'],
                 color=NAVY, edgecolor='white')
for bar, val in zip(bars2, abandon_region['abandonment_rate_%']):
    ax2.text(val + 0.05, bar.get_y() + bar.get_height()/2, f'{val:.2f}%',
             va='center', fontsize=10, fontweight='bold', color=NAVY)
ax2.set_xlabel('Abandonment Rate (%)', fontsize=11)
ax2.set_title('Abandonment Rate by Region', fontsize=13, fontweight='bold', color=NAVY)

plt.tight_layout()
plt.savefig('abandonment_rates.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Section 5: Cost Dimension Analysis — Operational Costs and Financial Trends

**Objective:** Quantify rising operational costs, overtime dependency and missed revenue opportunities.


In [ ]:
# ── Cost Overview ─────────────────────────────────────────────────────────────
print("=" * 60)
print("COST DIMENSION SUMMARY (Nov 2024 to Oct 2025)")
print("=" * 60)
print(f"Total Annual Cost:         ${cost['total_cost_usd'].sum():,.2f}")
print(f"Average Cost Per Contact:  ${cost['cost_per_contact_usd'].mean():,.2f}")
print(f"Cost Nov 2024:             ${cost_sorted['total_cost_usd'].iloc[0]:,.2f}")
print(f"Cost Oct 2025:             ${cost_sorted['total_cost_usd'].iloc[-1]:,.2f}")
cost_growth = (cost_sorted['total_cost_usd'].iloc[-1] - cost_sorted['total_cost_usd'].iloc[0]) / cost_sorted['total_cost_usd'].iloc[0] * 100
print(f"Cost Growth:               +{cost_growth:.2f}%")
print(f"\nTotal Overtime Hours:      {cost['overtime_hours'].sum():,}")
ot_growth = (cost_sorted['overtime_hours'].iloc[-1] - cost_sorted['overtime_hours'].iloc[0]) / cost_sorted['overtime_hours'].iloc[0] * 100
print(f"Overtime Nov 2024:         {cost_sorted['overtime_hours'].iloc[0]:,} hours")
print(f"Overtime Oct 2025:         {cost_sorted['overtime_hours'].iloc[-1]:,} hours")
print(f"Overtime Growth:           +{ot_growth:.2f}%")
print(f"\nTotal Missed Revenue:      ${cost['missed_revenue_usd'].sum():,.2f}")
print(f"Total Missed Upsells:      {cost['missed_upsell_opportunities'].sum():,}")
upsell_growth = (cost_sorted['missed_upsell_opportunities'].iloc[-1] - cost_sorted['missed_upsell_opportunities'].iloc[0]) / cost_sorted['missed_upsell_opportunities'].iloc[0] * 100
print(f"Missed Upsell Growth:      +{upsell_growth:.2f}%")


In [ ]:
# ── Cost and Volume Trend Charts ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('GSH Cost Dimension Analysis — Nov 2024 to Oct 2025', 
             fontsize=16, fontweight='bold', color=NAVY, y=1.01)

# Chart 1: Total Cost Trend
ax1 = axes[0, 0]
ax1.plot(cost_sorted['month'], cost_sorted['total_cost_usd'], 
         color=RED, linewidth=2.5, marker='o', markersize=5)
ax1.fill_between(cost_sorted['month'], cost_sorted['total_cost_usd'], alpha=0.1, color=RED)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))
ax1.set_title('Total Monthly Operational Cost', fontweight='bold', color=NAVY)
ax1.tick_params(axis='x', rotation=45)
ax1.set_ylabel('Cost (USD)')

# Chart 2: Cost Per Contact Trend
ax2 = axes[0, 1]
ax2.plot(cost_sorted['month'], cost_sorted['cost_per_contact_usd'],
         color=NAVY, linewidth=2.5, marker='s', markersize=5)
ax2.fill_between(cost_sorted['month'], cost_sorted['cost_per_contact_usd'], alpha=0.1, color=NAVY)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x:.2f}'))
ax2.set_title('Cost Per Contact Trend', fontweight='bold', color=NAVY)
ax2.tick_params(axis='x', rotation=45)
ax2.set_ylabel('Cost Per Contact (USD)')

# Chart 3: Overtime Hours Trend
ax3 = axes[1, 0]
ax3.bar(range(len(cost_sorted)), cost_sorted['overtime_hours'], 
        color=ORANGE, edgecolor='white')
ax3.set_xticks(range(len(cost_sorted)))
ax3.set_xticklabels(cost_sorted['month'].dt.strftime('%b-%y'), rotation=45, ha='right')
ax3.set_title('Monthly Overtime Hours', fontweight='bold', color=NAVY)
ax3.set_ylabel('Overtime Hours')

# Chart 4: Missed Revenue Trend
ax4 = axes[1, 1]
ax4.bar(range(len(cost_sorted)), cost_sorted['missed_revenue_usd'],
        color=RED, edgecolor='white', alpha=0.8)
ax4.set_xticks(range(len(cost_sorted)))
ax4.set_xticklabels(cost_sorted['month'].dt.strftime('%b-%y'), rotation=45, ha='right')
ax4.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))
ax4.set_title('Monthly Missed Revenue', fontweight='bold', color=NAVY)
ax4.set_ylabel('Missed Revenue (USD)')

plt.tight_layout()
plt.savefig('cost_dimension_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Turnover and Utilisation Trends ──────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(cost_sorted['month'], cost_sorted['agent_turnover_rate'] * 100,
         color=RED, linewidth=2.5, marker='o', markersize=6)
ax1.fill_between(cost_sorted['month'], cost_sorted['agent_turnover_rate'] * 100,
                 alpha=0.1, color=RED)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x:.1f}%'))
ax1.set_title('Agent Turnover Rate Trend', fontsize=13, fontweight='bold', color=NAVY)
ax1.tick_params(axis='x', rotation=45)
ax1.set_ylabel('Turnover Rate (%)')

ax2.plot(cost_sorted['month'], cost_sorted['agent_utilization_rate'] * 100,
         color=MID_BLUE, linewidth=2.5, marker='s', markersize=6)
ax2.fill_between(cost_sorted['month'], cost_sorted['agent_utilization_rate'] * 100,
                 alpha=0.1, color=MID_BLUE)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x:.1f}%'))
ax2.set_title('Agent Utilisation Rate Trend', fontsize=13, fontweight='bold', color=NAVY)
ax2.tick_params(axis='x', rotation=45)
ax2.set_ylabel('Utilisation Rate (%)')

plt.tight_layout()
plt.savefig('turnover_utilisation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Turnover Nov 2024: {cost_sorted['agent_turnover_rate'].iloc[0]*100:.1f}%")
print(f"Turnover Oct 2025: {cost_sorted['agent_turnover_rate'].iloc[-1]*100:.1f}%")
print(f"\nUtilisation Nov 2024: {cost_sorted['agent_utilization_rate'].iloc[0]*100:.1f}%")
print(f"Utilisation Oct 2025: {cost_sorted['agent_utilization_rate'].iloc[-1]*100:.1f}%")
print("\nKey Finding: As volume grows, utilisation drops meaning agents are being stretched thinner.")


---
## Section 6: Automation Opportunity Analysis

**Objective:** Identify the proportion of repetitive inquiries suitable for AI concierge automation, and validate that automation is safe based on escalation patterns.


In [ ]:
# ── Repetitive vs Non-Repetitive Overview ─────────────────────────────────────
rep_total = agent['is_repetitive'].sum()
non_rep_total = (~agent['is_repetitive']).sum()
rep_rate = rep_total / len(agent) * 100

print("=" * 60)
print("AUTOMATION OPPORTUNITY OVERVIEW")
print("=" * 60)
print(f"Total Tickets:              {len(agent):,}")
print(f"Repetitive (Automatable):   {rep_total:,} ({rep_rate:.2f}%)")
print(f"Non-Repetitive (Human):     {non_rep_total:,} ({100-rep_rate:.2f}%)")

# By category
cat_analysis = agent.groupby(['inquiry_category', 'is_repetitive'])['ticket_id'].count().unstack(fill_value=0)
cat_analysis.columns = ['Non_Repetitive', 'Repetitive']
cat_analysis['Total'] = cat_analysis.sum(axis=1)
cat_analysis['Pct_of_Total'] = (cat_analysis['Total'] / len(agent) * 100).round(2)
cat_analysis['Type'] = cat_analysis.apply(
    lambda r: 'Repetitive' if r['Repetitive'] > 0 else 'Non Repetitive', axis=1
)
cat_analysis = cat_analysis.sort_values('Total', ascending=False)

print("\nBREAKDOWN BY INQUIRY CATEGORY")
print(cat_analysis[['Repetitive', 'Non_Repetitive', 'Total', 'Pct_of_Total', 'Type']].to_string())


In [ ]:
# ── Visualise Automation Opportunity ─────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))

# Donut chart
sizes = [rep_rate, 100 - rep_rate]
labels = [f'Repetitive\n(Automatable)\n{rep_rate:.1f}%', f'Non-Repetitive\n(Human Required)\n{100-rep_rate:.1f}%']
colours_donut = [MID_BLUE, GREY]
wedges, texts = ax1.pie(sizes, labels=labels, colors=colours_donut,
                         startangle=90, wedgeprops=dict(width=0.6))
for text in texts:
    text.set_fontsize(11)
    text.set_fontweight('bold')
ax1.set_title('Repetitive vs Non-Repetitive Inquiries', fontsize=13, fontweight='bold', color=NAVY)

# Bar chart by category
colours_bar = [MID_BLUE if t == 'Repetitive' else GREY for t in cat_analysis['Type']]
bars = ax2.barh(cat_analysis.index, cat_analysis['Total'], color=colours_bar, edgecolor='white')
ax2.set_xlabel('Total Tickets', fontsize=11)
ax2.set_title('Ticket Volume by Inquiry Category and Type', fontsize=13, fontweight='bold', color=NAVY)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=MID_BLUE, label='Repetitive (Automatable)'),
                   Patch(facecolor=GREY, label='Non-Repetitive (Human Required)')]
ax2.legend(handles=legend_elements, fontsize=10)

plt.tight_layout()
plt.savefig('automation_opportunity.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Knowledge Base Search Time by Category ────────────────────────────────────
kb_time = knowledge.groupby('inquiry_category')['kb_search_time_min'].mean().reset_index()
kb_time.columns = ['inquiry_category', 'avg_kb_search_time']
kb_time = kb_time.sort_values('avg_kb_search_time', ascending=False)

repetitive_cats = cat_analysis[cat_analysis['Type'] == 'Repetitive'].index.tolist()
kb_time['Type'] = kb_time['inquiry_category'].apply(
    lambda x: 'Repetitive' if x in repetitive_cats else 'Non Repetitive'
)

print("KB SEARCH TIME BY CATEGORY")
print(kb_time.to_string(index=False))
print(f"\nOverall average KB search time: {knowledge['kb_search_time_min'].mean():.2f} minutes")

rep_kb = kb_time[kb_time['Type'] == 'Repetitive']['avg_kb_search_time'].mean()
non_rep_kb = kb_time[kb_time['Type'] == 'Non Repetitive']['avg_kb_search_time'].mean()
print(f"Repetitive categories average: {rep_kb:.2f} mins")
print(f"Non-repetitive categories average: {non_rep_kb:.2f} mins")

total_kb_saved = rep_total * knowledge['kb_search_time_min'].mean()
print(f"\nPotential KB search time saved by automation: {total_kb_saved:,.0f} minutes = {total_kb_saved/60:,.0f} hours")


In [ ]:
# ── Escalation Rate by Category ───────────────────────────────────────────────
escalation = knowledge.groupby('inquiry_category').agg(
    Total=('was_escalated', 'count'),
    Escalated=('was_escalated', 'sum')
).reset_index()
escalation['Escalation_Rate_%'] = (escalation['Escalated'] / escalation['Total'] * 100).round(2)
escalation['Type'] = escalation['inquiry_category'].apply(
    lambda x: 'Repetitive' if x in repetitive_cats else 'Non Repetitive'
)
escalation = escalation.sort_values('Escalation_Rate_%', ascending=False)

print("ESCALATION RATE BY CATEGORY")
print(escalation[['inquiry_category', 'Escalation_Rate_%', 'Type']].to_string(index=False))
print(f"\nOverall escalation rate: {knowledge['was_escalated'].mean()*100:.2f}%")

rep_esc = escalation[escalation['Type'] == 'Repetitive']['Escalation_Rate_%'].mean()
non_rep_esc = escalation[escalation['Type'] == 'Non Repetitive']['Escalation_Rate_%'].mean()
print(f"Repetitive categories avg escalation: {rep_esc:.2f}%")
print(f"Non-repetitive categories avg escalation: {non_rep_esc:.2f}%")

fig, ax = plt.subplots(figsize=(12, 7))
colours_esc = [RED if t == 'Non Repetitive' else GREEN for t in escalation['Type']]
bars = ax.barh(escalation['inquiry_category'], escalation['Escalation_Rate_%'],
               color=colours_esc, edgecolor='white')
for bar, val in zip(bars, escalation['Escalation_Rate_%']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.1f}%',
            va='center', fontsize=9, color=NAVY)
ax.set_xlabel('Escalation Rate (%)', fontsize=11)
ax.set_title('Escalation Rate by Inquiry Category\n(Red = Non-Repetitive, Green = Repetitive/Automatable)',
             fontsize=13, fontweight='bold', color=NAVY)
ax.axvline(x=20, color=GOLD, linestyle='--', linewidth=1.5, label='20% threshold (safe for automation)')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('escalation_by_category.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nKey Finding: Repetitive categories escalate at 9-18% vs 58-68% for non-repetitive. Automation is safe.")


---
## Section 7: Financial Impact Model

**Objective:** Translate operational inefficiencies into a measurable financial business case for the AI Travel Concierge.

### Model Assumptions

| Assumption | Value | Source |
|------------|-------|--------|
| Automation Coverage Rate | 80% | Decagon / Ada benchmarks (up to 80-83% deflection) |
| Automation Accuracy | 95% | Fini Labs (2026) — 98% accuracy across 2M queries |
| Overtime Reduction | 40% | McKinsey lower bound (40-50% interaction reduction) |
| Revenue Recovery | 30% | Gartner 14% productivity boost freeing agents for upsells |
| Implementation Cost | $500,000 | Standard enterprise AI concierge estimate |


In [ ]:
# ── Financial Impact Calculations ────────────────────────────────────────────
print("=" * 60)
print("FINANCIAL IMPACT MODEL — AI TRAVEL CONCIERGE")
print("=" * 60)

# Current State
total_inquiries = len(agent)
repetitive_volume = rep_total
repetitive_rate = rep_rate / 100
avg_cost_per_contact = cost['cost_per_contact_usd'].mean()
total_annual_cost = cost['total_cost_usd'].sum()
total_overtime = cost['overtime_hours'].sum()
total_missed_revenue = cost['missed_revenue_usd'].sum()

print("\nCURRENT STATE METRICS")
print(f"  Total Annual Inquiries:     {total_inquiries:,}")
print(f"  Repetitive Inquiry Rate:    {repetitive_rate*100:.2f}%")
print(f"  Repetitive Inquiries:       {repetitive_volume:,}")
print(f"  Average Cost Per Contact:   ${avg_cost_per_contact:.2f}")
print(f"  Total Annual Cost:          ${total_annual_cost:,.2f}")
print(f"  Total Overtime Hours:       {total_overtime:,}")
print(f"  Total Missed Revenue:       ${total_missed_revenue:,.2f}")

# Automation assumptions
automation_coverage = 0.80
inquiries_automated = repetitive_volume * automation_coverage
overtime_rate_per_hour = 35

# Projected savings
cost_savings = inquiries_automated * avg_cost_per_contact
overtime_reduction = total_overtime * 0.40 * overtime_rate_per_hour
revenue_recovery = total_missed_revenue * 0.30
total_savings = cost_savings + overtime_reduction + revenue_recovery

# Net impact
implementation_cost = 500000
net_benefit = total_savings - implementation_cost
roi = net_benefit / implementation_cost * 100

print("\nPROJECTED SAVINGS")
print(f"  Inquiries Automated:        {inquiries_automated:,.0f}")
print(f"  Cost Savings:               ${cost_savings:,.2f}")
print(f"  Overtime Reduction:         ${overtime_reduction:,.2f}")
print(f"  Revenue Recovery:           ${revenue_recovery:,.2f}")
print(f"  Total Projected Savings:    ${total_savings:,.2f}")

print("\nNET IMPACT")
print(f"  Total Projected Savings:    ${total_savings:,.2f}")
print(f"  Implementation Cost:        ${implementation_cost:,.2f}")
print(f"  Net Financial Benefit:      ${net_benefit:,.2f}")
print(f"  Return on Investment (ROI): {roi:.2f}%")
print(f"\n  For every $1 invested GSH gets ${1 + roi/100:.2f} back")


In [ ]:
# ── Financial Impact Visualisation ───────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Waterfall chart
categories = ['Cost Savings', 'Overtime\nReduction', 'Revenue\nRecovery', 'Implementation\nCost', 'Net Benefit']
values = [cost_savings, overtime_reduction, revenue_recovery, -implementation_cost, net_benefit]
colours_wf = [GREEN, GREEN, GREEN, RED, MID_BLUE]

bars = ax1.bar(categories, [abs(v) for v in values], color=colours_wf, edgecolor='white', width=0.6)
for bar, val in zip(bars, values):
    label = f'${abs(val)/1e6:.2f}M' if abs(val) >= 1e6 else f'${abs(val)/1e3:.0f}K'
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30000,
             label, ha='center', va='bottom', fontweight='bold', fontsize=10, color=NAVY)
ax1.set_title('Financial Impact Breakdown (USD)', fontsize=13, fontweight='bold', color=NAVY)
ax1.set_ylabel('Amount (USD)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=GREEN, label='Savings'),
                   Patch(facecolor=RED, label='Cost'),
                   Patch(facecolor=MID_BLUE, label='Net Benefit')]
ax1.legend(handles=legend_elements)

# ROI comparison chart
scenarios = ['Do Nothing\n(Year 1)', 'AI Concierge\nInvestment', 'Net Benefit']
scenario_vals = [total_annual_cost, implementation_cost, net_benefit]
scenario_cols = [RED, ORANGE, GREEN]
bars2 = ax2.bar(scenarios, scenario_vals, color=scenario_cols, edgecolor='white', width=0.5)
for bar, val in zip(bars2, scenario_vals):
    label = f'${val/1e6:.2f}M'
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50000,
             label, ha='center', va='bottom', fontweight='bold', fontsize=11, color=NAVY)
ax2.set_title(f'Investment vs Return Analysis\nROI: {roi:.1f}%', 
              fontsize=13, fontweight='bold', color=NAVY)
ax2.set_ylabel('Amount (USD)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))

plt.tight_layout()
plt.savefig('financial_impact_model.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Section 8: Performance Benchmarks and Measurement Framework

**Objective:** Establish baseline KPIs and post-automation targets as reference points for evaluating the AI concierge impact.


In [ ]:
# ── Performance Benchmarks Table ─────────────────────────────────────────────
benchmarks = pd.DataFrame({
    'KPI': [
        'Average Response Time',
        'SLA Breach Rate',
        'Peak Hour Breach Rate',
        'Off-Peak Breach Rate',
        'Global Abandonment Rate',
        'Cost Per Contact (USD)',
        'Total Annual Cost (USD)',
        'Total Overtime Hours',
        'Agent Turnover Rate',
        'Total Missed Revenue (USD)',
        'Repetitive Inquiry Rate',
        'Overall Escalation Rate',
        'Avg KB Search Time (mins)',
        'Avg Time Saved if Automated'
    ],
    'Baseline_Value': [
        f'{agent["Response_Time_Min"].mean():.2f} mins',
        f'{breach_rate:.2f}%',
        f'{peak_hours:.2f}%',
        f'{offpeak_hours:.2f}%',
        f'{inquiry["abandonment_rate"].mean()*100:.2f}%',
        f'${avg_cost_per_contact:.2f}',
        f'${total_annual_cost:,.2f}',
        f'{total_overtime:,}',
        f'{cost["agent_turnover_rate"].iloc[-1]*100:.1f}%',
        f'${total_missed_revenue:,.2f}',
        f'{rep_rate:.2f}%',
        f'{knowledge["was_escalated"].mean()*100:.2f}%',
        f'{knowledge["kb_search_time_min"].mean():.2f} mins',
        f'{knowledge["time_saved_if_automated_min"].mean():.2f} mins'
    ],
    'Post_Automation_Target': [
        '4.02 mins', '30%', '50%', '35%', '5%',
        '$35.00', f'${total_annual_cost - cost_savings:,.2f}',
        f'{total_overtime * 0.60:,.0f}',
        '7%', f'${total_missed_revenue * 0.70:,.2f}',
        '60.11% (unchanged)', '15%', '1.50 mins', '7.73 mins'
    ],
    'Method': [
        'Calculated (60% improvement)',
        'Industry Benchmark (Gartner 2026)',
        'Industry Benchmark (McKinsey 2023)',
        'Calculated (40% improvement)',
        'Industry Benchmark (Forrester 2023)',
        'Industry Benchmark (IBM 2022)',
        'Calculated (cost savings applied)',
        'Calculated (40% reduction)',
        'Target reduction',
        'Calculated (30% recovery)',
        'Baseline data',
        'Target reduction',
        'Target with AI KB',
        'Baseline data'
    ]
})

print("PERFORMANCE BENCHMARKS")
print(benchmarks.to_string(index=False))


In [ ]:
# ── Measurement Framework ─────────────────────────────────────────────────────
measurement_framework = pd.DataFrame({
    'KPI': [
        'Average Response Time',
        'SLA Breach Rate',
        'Abandonment Rate',
        'Cost Per Contact',
        'Overtime Hours',
        'Customer CSAT',
        'Escalation Rate',
        'Missed Revenue'
    ],
    'Baseline': ['10.04 mins', '72%', '10.72%', '$55.16', '18,407/yr', '3.23/5', '31.96%', '$501,101'],
    'Target': ['4.02 mins', '30%', '5%', '$35.00', '11,044/yr', '4.00/5', '15%', '$350,771'],
    'Improvement': ['60% faster', '42pts lower', '53% reduction', '37% reduction', '40% reduction', '+23% uplift', '53% reduction', '30% recovery'],
    'Monitoring_Frequency': ['Daily', 'Daily', 'Daily', 'Weekly', 'Weekly', 'Weekly', 'Weekly', 'Monthly'],
    'Responsible_Team': ['Operations', 'Operations', 'Customer Service', 'Finance', 'HR', 'Customer Service', 'Operations', 'Finance'],
    'Review_Method': [
        'Live dashboard',
        'Automated alerts',
        'Queue monitoring',
        'Weekly cost report',
        'Workforce system',
        'Post-interaction surveys',
        'Escalation log review',
        'Revenue report'
    ]
})

print("POST-AUTOMATION MEASUREMENT FRAMEWORK")
print(measurement_framework.to_string(index=False))


In [ ]:
# ── Before vs After Comparison Visual ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))

kpis = ['SLA Breach\nRate %', 'Abandonment\nRate %', 'Escalation\nRate %', 
        'Peak Hour\nBreach %', 'Agent\nTurnover %']
baseline_vals = [breach_rate, 
                 inquiry['abandonment_rate'].mean()*100,
                 knowledge['was_escalated'].mean()*100,
                 peak_hours,
                 cost['agent_turnover_rate'].iloc[-1]*100]
target_vals = [30, 5, 15, 50, 7]

x = np.arange(len(kpis))
width = 0.35

bars1 = ax.bar(x - width/2, baseline_vals, width, label='Baseline (Current State)', 
               color=RED, alpha=0.85, edgecolor='white')
bars2 = ax.bar(x + width/2, target_vals, width, label='Target (Post-Automation)',
               color=GREEN, alpha=0.85, edgecolor='white')

for bar, val in zip(bars1, baseline_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9.5, fontweight='bold', color=NAVY)
for bar, val in zip(bars2, target_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9.5, fontweight='bold', color=NAVY)

ax.set_xticks(x)
ax.set_xticklabels(kpis, fontsize=11)
ax.set_ylabel('Rate (%)', fontsize=11)
ax.set_title('Pre vs Post-Automation Performance Targets', fontsize=14, fontweight='bold', color=NAVY, pad=15)
ax.legend(fontsize=11)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig('benchmarks_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Section 9: Executive Summary Dashboard

A comprehensive summary of all key findings and recommendations.


In [ ]:
# ── Executive Summary Dashboard ──────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor('#F2F2F2')

# Title
fig.text(0.5, 0.97, 'GrandStay Hospitality Group', ha='center', fontsize=22, 
         fontweight='bold', color=NAVY)
fig.text(0.5, 0.945, 'Hospitality Operations Intelligence Initiative — Executive Summary', 
         ha='center', fontsize=14, color=GOLD)
fig.text(0.5, 0.925, f'Based on {len(agent):,} ticket records | Nov 2024 to Oct 2025 | May 2026', 
         ha='center', fontsize=11, color='#64748B', style='italic')

# KPI Cards row
kpi_data = [
    ('Total Tickets', f'{len(agent):,}', MID_BLUE),
    ('SLA Breach Rate', f'{breach_rate:.2f}%', RED),
    ('Avg Response Time', f'{agent["Response_Time_Min"].mean():.2f} mins', NAVY),
    ('Total Annual Cost', f'${total_annual_cost/1e6:.2f}M', RED),
    ('Repetitive Rate', f'{rep_rate:.2f}%', MID_BLUE),
    ('AI Concierge ROI', f'{roi:.1f}%', GREEN),
    ('Net Benefit', f'${net_benefit/1e6:.2f}M', GREEN),
    ('Missed Revenue', f'${total_missed_revenue/1e3:.0f}K', ORANGE)
]

for i, (label, value, colour) in enumerate(kpi_data):
    ax_kpi = fig.add_axes([0.03 + i * 0.122, 0.84, 0.108, 0.07])
    ax_kpi.set_facecolor(colour)
    ax_kpi.text(0.5, 0.65, value, ha='center', va='center', fontsize=14,
                fontweight='bold', color='white', transform=ax_kpi.transAxes)
    ax_kpi.text(0.5, 0.2, label, ha='center', va='center', fontsize=8,
                color='white', transform=ax_kpi.transAxes)
    ax_kpi.set_xticks([])
    ax_kpi.set_yticks([])

# Chart 1: SLA by Channel
ax1 = fig.add_axes([0.03, 0.47, 0.28, 0.33])
ch_colours = [RED, ORANGE, YELLOW, GREEN]
cb = ax1.barh(channel_breach['channel'], channel_breach['Breach_Rate_%'], color=ch_colours)
for bar, val in zip(cb, channel_breach['Breach_Rate_%']):
    ax1.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}%',
             va='center', fontsize=9, fontweight='bold')
ax1.set_title('SLA Breach Rate by Channel', fontweight='bold', color=NAVY, fontsize=11)
ax1.set_xlim(0, 105)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Chart 2: Cost Trend
ax2 = fig.add_axes([0.36, 0.47, 0.28, 0.33])
ax2.plot(range(len(cost_sorted)), cost_sorted['total_cost_usd'], 
         color=RED, linewidth=2.5, marker='o', markersize=4)
ax2.fill_between(range(len(cost_sorted)), cost_sorted['total_cost_usd'], alpha=0.15, color=RED)
ax2.set_xticks(range(len(cost_sorted)))
ax2.set_xticklabels(cost_sorted['month'].dt.strftime('%b-%y'), rotation=45, ha='right', fontsize=7)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x/1e3:.0f}K'))
ax2.set_title('Monthly Operational Cost Trend', fontweight='bold', color=NAVY, fontsize=11)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Chart 3: Automation Split
ax3 = fig.add_axes([0.69, 0.47, 0.28, 0.33])
sizes = [rep_rate, 100 - rep_rate]
wedges, texts, autotexts = ax3.pie(sizes, colors=[MID_BLUE, GREY],
                                    autopct='%1.1f%%', startangle=90,
                                    wedgeprops=dict(width=0.6))
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight('bold')
ax3.set_title('Repetitive vs Non-Repetitive', fontweight='bold', color=NAVY, fontsize=11)
ax3.legend(['Repetitive (Automatable)', 'Non-Repetitive'], loc='lower center', fontsize=8)

# Chart 4: Hour breach
ax4 = fig.add_axes([0.03, 0.1, 0.44, 0.32])
colours_h = [RED if (8 <= h <= 11) or (15 <= h <= 19) else MID_BLUE 
             for h in hour_breach['hour_of_day']]
ax4.bar(hour_breach['hour_of_day'], hour_breach['Breach_Rate_%'], color=colours_h, edgecolor='white')
ax4.axhline(y=breach_rate, color=GOLD, linestyle='--', linewidth=2)
ax4.set_xlabel('Hour of Day')
ax4.set_ylabel('SLA Breach Rate (%)')
ax4.set_title('SLA Breach Rate by Hour — Peak Windows at 8-11 and 15-19', fontweight='bold', color=NAVY, fontsize=11)
ax4.set_xticks(range(0, 24))
ax4.spines['top'].set_visible(False)
ax4.spines['right'].set_visible(False)

# Chart 5: Financial Impact
ax5 = fig.add_axes([0.54, 0.1, 0.43, 0.32])
fin_cats = ['Cost\nSavings', 'Overtime\nReduction', 'Revenue\nRecovery', 'Implementation\nCost', 'Net\nBenefit']
fin_vals = [cost_savings, overtime_reduction, revenue_recovery, implementation_cost, net_benefit]
fin_cols = [GREEN, GREEN, GREEN, RED, MID_BLUE]
bars_f = ax5.bar(fin_cats, fin_vals, color=fin_cols, edgecolor='white')
for bar, val in zip(bars_f, fin_vals):
    label = f'${val/1e6:.2f}M' if val >= 1e6 else f'${val/1e3:.0f}K'
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20000,
             label, ha='center', va='bottom', fontweight='bold', fontsize=9, color=NAVY)
ax5.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))
ax5.set_title(f'Financial Impact — ROI: {roi:.1f}%', fontweight='bold', color=NAVY, fontsize=11)
ax5.spines['top'].set_visible(False)
ax5.spines['right'].set_visible(False)

plt.savefig('executive_summary_dashboard.png', dpi=150, bbox_inches='tight', 
            facecolor=fig.get_facecolor())
plt.show()
print("Executive summary dashboard saved.")


---
## Section 10: Conclusions and Strategic Recommendations

### Key Findings Summary

| Dimension | Finding | Impact |
|-----------|---------|--------|
| **Time** | 71.97% overall SLA breach rate across all 7 regions | Systemic not regional — automation required |
| **Time** | Phone 84.82% and Chat 77.06% breach rates | Highest volume channels performing worst |
| **Time** | Two peak windows at hours 8-11 and 15-19 with 82-83% breach rate | 30 percentage point spike during peaks |
| **Scale** | 39% inquiry volume growth in 12 months | Operation cannot scale manually |
| **Scale** | 10.72% global abandonment rate — 11,810 guests gave up | Significant revenue and satisfaction risk |
| **Cost** | Total cost grew 54.82% in 12 months | Costs rising faster than volume |
| **Cost** | 18,407 overtime hours annually growing 52.5% | Unsustainable workforce pressure |
| **Cost** | $501,101 in missed revenue with upsells growing 163% | Agents too busy for revenue opportunities |
| **Automation** | 60.11% of inquiries are repetitive and automatable | Clear AI concierge opportunity |
| **Automation** | Repetitive categories escalate at only 9-18% | Automation is safe |
| **Financial** | $3.09M net benefit with 618% ROI | Compelling business case |

### Strategic Recommendations

1. **Deploy AI Travel Concierge** (Immediate) — Handle the 12 automatable categories representing 60.11% of all tickets. Target Chat and Phone channels first. Expected ROI 618%.

2. **Redesign Peak Hour Staffing** (Short-term) — Introduce dynamic staffing for the 8-11am and 3-7pm windows where breach rates spike to 83%.

3. **Centralise Knowledge Management** (Short-term) — Implement unified knowledge base to reduce 3.65-minute average KB search time and recover 4,388 agent hours annually.

4. **Reduce Overtime Dependency** (Medium-term) — Target 40% overtime reduction from 18,407 to 11,044 hours using AI absorption of repetitive volume.

5. **Implement Performance Monitoring** (Ongoing) — Deploy measurement framework with daily SLA and response time monitoring, weekly cost and CSAT reviews, monthly revenue tracking.


In [ ]:
# ── Final Summary Statistics ──────────────────────────────────────────────────
print("=" * 70)
print("GRANDSTAY HOSPITALITY GROUP — FINAL ANALYSIS SUMMARY")
print("=" * 70)

print("\nDATA PROCESSED")
print(f"  Agent Performance Logs:    {len(agent):,} tickets")
print(f"  Cost Efficiency Metrics:   {len(cost):,} monthly records")
print(f"  Inquiry Volume Patterns:   {len(inquiry):,} hourly records")
print(f"  Knowledge Escalation Logs: {len(knowledge):,} records")

print("\nKEY FINDINGS")
print(f"  Overall SLA Breach Rate:   {breach_rate:.2f}%")
print(f"  Worst Channel (Phone):     {channel_breach.set_index('channel').loc['Phone', 'Breach_Rate_%']:.2f}%")
print(f"  Peak Hour Breach Rate:     {peak_hours:.2f}%")
print(f"  Total Annual Cost:         ${total_annual_cost:,.2f}")
print(f"  Cost Growth (12 months):   +{cost_growth:.2f}%")
print(f"  Overtime Hours:            {total_overtime:,}")
print(f"  Missed Revenue:            ${total_missed_revenue:,.2f}")
print(f"  Global Abandonment Rate:   {inquiry['abandonment_rate'].mean()*100:.2f}%")
print(f"  Repetitive Inquiry Rate:   {rep_rate:.2f}%")
print(f"  Automatable Tickets:       {rep_total:,}")

print("\nFINANCIAL MODEL")
print(f"  Inquiries Automated:       {inquiries_automated:,.0f}")
print(f"  Cost Savings:              ${cost_savings:,.2f}")
print(f"  Overtime Reduction:        ${overtime_reduction:,.2f}")
print(f"  Revenue Recovery:          ${revenue_recovery:,.2f}")
print(f"  Total Projected Savings:   ${total_savings:,.2f}")
print(f"  Implementation Cost:       ${implementation_cost:,.2f}")
print(f"  Net Financial Benefit:     ${net_benefit:,.2f}")
print(f"  Return on Investment:      {roi:.2f}%")

print("\nCONCLUSION")
print(f"  The data confirms that GSH's guest support operation is under")
print(f"  severe systemic pressure. Implementing an AI Travel Concierge")
print(f"  to handle the {rep_total:,} repetitive annual inquiries ({rep_rate:.1f}%)")
print(f"  would deliver a net benefit of ${net_benefit:,.2f} with a")
print(f"  {roi:.1f}% return on a ${implementation_cost:,} investment.")
print("=" * 70)


---

## References

- Decagon (2026). AI Concierge Platform Case Studies. decagon.ai
- Fini Labs (2026). AI Customer Support ROI: 7 Platforms Ranked. usefini.com
- Gartner (2026). Predicts: Agentic AI to resolve 80% of common issues by 2029. groovehq.com
- Gartner (2026). Self-service benchmarked at $1.84 vs $13.50 for human agents. lorikeetcx.ai
- McKinsey Global Institute (2023). AI deployments reduce total interactions by 40-50%. fin.ai
- NextPhone (2026). 75 AI Customer Service Statistics 2026. getnextphone.com
- Klarna (2024). AI assistant handles two-thirds of customer service chats. groovehq.com
- Ada (2026). Up to 83% automated resolution rates. usefini.com

---

*Notebook prepared by Oba Adelore | Amdari Analytics Programme | May 2026*
